In [5]:
import pandas as pd
import numpy as np
from functions import *

BASE_DATE = pd.Timestamp("2018-01-01")

exclude_types = [
    'Clubs', 'Muds', 'Hotels', 'Clubs Free',
    "In House Demo's", 'Dealers demo',
    'VIP_CNE', 'Bein Companies PV',
    'Temp', 'Temp OSN'
]

HW_COLS = [
    'Old Decoder Number',
    'New Decoder Number',
    'Old Smartcard Number',
    'New Smartcard Number'
]

date = datetime.now().strftime('%d-%m-%Y')
base_dir_format = datetime.now().strftime('%d.%m.%y')
zip_file_name_format = datetime.now().strftime('%d-%m-%Y')
base_dir = f's:\\{base_dir_format}'

In [ ]:

bein_original_path = search_files(base_dir,'BEINDATANEWRPT')[0]
swap_original_path = search_files(base_dir,'SWAPRPT')[0]
swap_details_path = search_files(base_dir,'SWAPWTDTLS')[0]


# print(bein_original_path)
# print(swap_original_path)
# print(swap_details_path)

bein_original = pd.read_csv(
    bein_original_path,
    dtype="str"
)

swap_original = pd.read_csv(
   swap_original_path,
    dtype="str"
)

swap_details = pd.read_csv(swap_details_path, dtype='str')


s:\16.08.26\30749603_BEINDATANEWRPT.CSV
s:\16.08.26\30750612_SWAPRPT.CSV
s:\16.08.26\30744505_SWAPWTDTLS.CSV


In [7]:

hw_old = swap_details[['OLD_HW','OLD_TYPE']].rename(columns={'OLD_HW':'HW','OLD_TYPE':'HW TYPE'})
hw_new = swap_details[['NEW_HW','NEW_TYPE']].rename(columns={'NEW_HW':'HW','NEW_TYPE':'HW TYPE'})



bein_hw_dec = bein_original[['Decoder','Item Description STB']].rename(columns={'Decoder':'HW','Item Description STB':'HW TYPE'})
bein_hw_sc = bein_original[['Smart Card','Item Description SC']].rename(columns={'Smart Card':'HW','Item Description SC':'HW TYPE'})



hw_all_types = pd.concat([hw_old,hw_new,bein_hw_dec,bein_hw_sc])
hw_all_types = hw_all_types.drop_duplicates()
hw_all_types

,HW,HW TYPE
0,10137612601,beIN Smartcard 1000s
1,10388776071,beIN Smartcard 1000s
2,10110488391,beIN Smartcard 1000s
3,0308986213,beIN Decoder 1000s
4,42768237085,beIN Smartcard 1000s
...,...,...
3045693,42916623509,beIN Smartcard 1000s
3045696,42912373976,Smartcard 3030
3045704,42916939681,CNE JSC Premeium SC
3045705,42916818737,beIN Smartcard 1000s


In [8]:
swap_original["Swap Datetime"] = pd.to_datetime(
    swap_original["Swap Date"].str.split().str[0] + " " + swap_original["Swap Time"],
    format="%d/%m/%Y %I:%M:%S %p"
)

swap = (
    swap_original
    .loc[
        (swap_original["Item"] != "ART Smart Card") &
        (~swap_original["Subscriber Type"].isin(exclude_types))
    ]
    [
        [
            "Swap Datetime",
            "Subscriber Number",
            "Subscriber Type",
            "Replacement Number",
            "Item",
            "Old Serial Number",
            "New Serial Number",
        ]
    ]
    .sort_values(["Subscriber Number", "Swap Datetime"])
)

In [9]:


swap_multi_hw = (
    swap_original
    .loc[
        (swap_original["Item"] != "ART Smart Card") &
        (swap_original["Subscriber Type"].isin(exclude_types))
    ]
    [
        [
            "Swap Datetime",
            "Subscriber Number",
            "Subscriber Type",
            "Replacement Number",
            "Item",
            "Old Serial Number",
            "New Serial Number",
        ]
    ]
    .sort_values(["Subscriber Number", "Swap Datetime"])
)

In [10]:
swap_multi_hw

,Swap Datetime,Subscriber Number,Subscriber Type,Replacement Number,Item,Old Serial Number,New Serial Number
9023,2020-03-19 17:33:08,10396800,Clubs,8200,Decoder,0302825083,0319644679
29331,2023-11-06 14:48:50,10537973,Clubs,34419,Decoder,0319696874,0302738972
29332,2023-11-06 14:48:50,10537973,Clubs,34419,Smartcard,10389067264,10719456898
22641,2025-11-26 16:58:27,10537973,Clubs,43435,Decoder,0302738972,0360931030
25338,2020-07-26 14:27:39,10709659,Hotels,9558,Decoder,0303444151,0315446020
...,...,...,...,...,...,...,...
4664,2023-01-11 15:20:33,8983562,Hotels,30090,Decoder,0303495418,0297245066
60440,2023-01-15 16:27:39,8983562,Hotels,30157,Smartcard,42916864723,10719699349
52522,2023-01-17 15:05:16,8983562,Hotels,30198,Smartcard,10088424774,10719699307
18811,2021-08-24 12:29:18,999089,Hotels,18839,Smartcard,42768405583,10691105034


In [11]:
swap_sc = (
    swap.loc[swap['Item']=='Smartcard']
    .sort_values('Swap Datetime')
    .rename(columns={'Old Serial Number':'New Smartcard Number'})
    .drop(columns=["Replacement Number",'New Serial Number','Item'])
    .drop_duplicates('Subscriber Number',keep='first')
    )

swap_dec = (
    swap.loc[swap['Item']=='Decoder']
    .sort_values('Swap Datetime')
    .rename(columns={'Old Serial Number':'New Decoder Number'})
    .drop(columns=["Replacement Number",'New Serial Number','Item'])
    .drop_duplicates('Subscriber Number',keep='first')
    )



swap_base = pd.merge(left=swap_dec, right=swap_sc[['Subscriber Number','New Smartcard Number']], on='Subscriber Number',how='inner')

swap_base['Transaction Type'] = 'Base'


# 	Subscriber Number	Subscriber Type	Status	New Decoder Number	New Smartcard Number	Swap Datetime

In [12]:
swap_sc = (
    swap.loc[swap["Item"] == "Smartcard"]
    .rename(
        columns={
            "Old Serial Number": "Old Smartcard Number",
            "New Serial Number": "New Smartcard Number",
        }
    )
    .assign(
        **{
            "Old Decoder Number": pd.NA,
            "New Decoder Number": pd.NA,
        }
    )
)

swap_dec = (
    swap.loc[swap["Item"] == "Decoder"]
    .rename(
        columns={
            "Old Serial Number": "Old Decoder Number",
            "New Serial Number": "New Decoder Number",
        }
    )
    .assign(
        **{
            "Old Smartcard Number": pd.NA,
            "New Smartcard Number": pd.NA,
        }
    )
)

In [13]:
all_swaps = (
    pd.concat([swap_dec, swap_sc], ignore_index=True)
    .sort_values(["Subscriber Number", "Swap Datetime"])
)

In [14]:
bein_base = (
    bein_original[
        [
            "Customer Number",
            "Customer Type",
            "Status",
            "Decoder",
            "Smart Card",
        ]
    ]
    .sort_values('Status')
    .drop_duplicates("Customer Number")
)

bein_base = (
    bein_base.loc[
        ~bein_base["Customer Type"].isin(exclude_types)
    ]
    .rename(
        columns={
            "Customer Number": "Subscriber Number",
            "Customer Type": "Subscriber Type",
            "Decoder": "New Decoder Number",
            "Smart Card": "New Smartcard Number",
        }
    )
)

bein_base["Swap Datetime"] = BASE_DATE

In [15]:
def create_base(df, old_col, new_col, drop_cols):
    base = (
        df.sort_values(["Subscriber Number", "Swap Datetime"])
          .drop_duplicates("Subscriber Number", keep="first")
          .copy()
    )

    base["Swap Datetime"] = BASE_DATE

    return (
        base.drop(columns=drop_cols)
            .rename(columns={old_col: new_col})
    )

In [16]:
base_sc = create_base(
    swap_sc,
    "Old Smartcard Number",
    "New Smartcard Number",
    [
        "Replacement Number",
        "Item",
        "Old Decoder Number",
        "New Smartcard Number",
    ],
)

base_dec = create_base(
    swap_dec,
    "Old Decoder Number",
    "New Decoder Number",
    [
        "Replacement Number",
        "Item",
        "New Decoder Number",
        "Old Smartcard Number",
    ],
)

In [17]:
history = all_swaps.copy()

history[HW_COLS] = (
    history
    .groupby("Subscriber Number")[HW_COLS]
    .ffill()
)

In [18]:
history = (
    history
    .drop(columns=[
        "Replacement Number",
        "Item",
        "Old Smartcard Number",
        "Old Decoder Number",
    ])
    .drop_duplicates()
    .drop_duplicates(
        subset=[
            "Subscriber Number",
            "Swap Datetime",
            "New Decoder Number",
        ],
        keep="last",
    )
    .drop_duplicates(
        subset=[
            "Subscriber Number",
            "Swap Datetime",
            "New Smartcard Number",
        ],
        keep="last",
    )
)

In [19]:
def merge_history(history, base, column):
    history = pd.concat([base, history], ignore_index=True)
    history = history.sort_values(["Subscriber Number", "Swap Datetime"])
    history[column] = (
        history.groupby("Subscriber Number")[column].ffill()
    )
    return history.loc[history["Swap Datetime"] != BASE_DATE]

In [20]:
history = merge_history(
    history,
    base_sc,
    "New Smartcard Number",
)

history = merge_history(
    history,
    base_dec,
    "New Decoder Number",
)

In [21]:
history = pd.concat([bein_base, history], ignore_index=True)
history = history.sort_values(["Subscriber Number", "Swap Datetime"])

history["New Decoder Number"] = (
    history.groupby("Subscriber Number")["New Decoder Number"].ffill()
)

history["New Smartcard Number"] = (
    history.groupby("Subscriber Number")["New Smartcard Number"].ffill()
)

history = history.loc[history["Swap Datetime"] != BASE_DATE]

In [22]:
missing = ~bein_base["Subscriber Number"].isin(history["Subscriber Number"])
bein_base['Transaction Type'] = 'Base'
history['Transaction Type'] = 'Swap'

history = pd.concat(
    [history, bein_base.loc[missing]],
    ignore_index=True,
).drop(columns=['Status'])

In [23]:
history = history.loc[(~history['New Smartcard Number'].isna()) & (~history['New Decoder Number'].isna())]


In [24]:
history.loc[history['Subscriber Number']=='10709659']

# history.to_csv('swap timeline.csv', index=False)

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime,Transaction Type


In [25]:
history = pd.concat([history,swap_base]).sort_values(['Swap Datetime','Subscriber Number'])

In [26]:
def encrypt_numbers(x):
    return x+1014

history = history.rename(columns={'New Decoder Number':'Decoder Number','New Smartcard Number':'Smartcard Number'})

history = history[['Swap Datetime','Subscriber Number', 'Subscriber Type', 'Decoder Number',
       'Smartcard Number',  'Transaction Type']]


history_merged_hw_types = pd.merge(left=history,right=hw_all_types, left_on='Decoder Number' , right_on='HW',how='left').rename(columns={'HW TYPE':"Decoder Type"}).drop(columns='HW')
history_merged_hw_types = pd.merge(left=history_merged_hw_types,right=hw_all_types, left_on='Smartcard Number' , right_on='HW',how='left').rename(columns={'HW TYPE':"Smartcard Type"}).drop(columns='HW')

history = history_merged_hw_types

history['Subscriber Number'] = pd.to_numeric(history['Subscriber Number']).apply(encrypt_numbers)
history['Decoder Number'] = pd.to_numeric(history['Decoder Number']).apply(encrypt_numbers)
history['Smartcard Number'] = pd.to_numeric(history['Smartcard Number']).apply(encrypt_numbers)


In [27]:

history['Swap Datetime'] = history['Swap Datetime'].dt.date
history = history.drop_duplicates(subset=['Swap Datetime','Subscriber Number','Transaction Type'], keep='last')

history = history[['Swap Datetime', 'Subscriber Number', 
       'Decoder Number', 'Decoder Type', 'Smartcard Number', 
        'Smartcard Type','Transaction Type',]]



In [28]:

history.loc[history['Subscriber Number']== 18996823+1014].sort_values('Swap Datetime').sort_values(['Transaction Type','Swap Datetime'])

,Swap Datetime,Subscriber Number,Decoder Number,Decoder Type,Smartcard Number,Smartcard Type,Transaction Type
825205,2022-01-25,18997837,360916453,beIN 4k,10676234567,CNE V7 Card,Base
825204,2022-01-25,18997837,352015329,Humax C1 Decoder,10708898472,CNE V7 Card,Swap
830360,2022-09-08,18997837,352015329,Humax C1 Decoder,10719580984,CNE V7 Card,Swap
836717,2023-06-08,18997837,351880484,Humax C1 Decoder,10719580984,CNE V7 Card,Swap
838632,2023-11-07,18997837,349921221,Humax C1 Decoder,10719580984,CNE V7 Card,Swap
840267,2024-02-10,18997837,349933847,Humax C1 Decoder,10719580984,CNE V7 Card,Swap


In [29]:

history.to_csv('Swap Timeline.csv', index =False)

In [30]:
g = history.groupby('Subscriber Number').agg(count=('Subscriber Number','count')).reset_index()
g.sort_values('count',ascending=False)

,Subscriber Number,count
499645,18579276,6
58176,13980115,6
618079,18997837,6
598377,18976108,6
629710,19010842,6
...,...,...
283549,16308752,1
283550,16308761,1
283551,16308770,1
283552,16308779,1


In [31]:
history.columns

Index(['Swap Datetime', 'Subscriber Number', 'Decoder Number', 'Decoder Type',
       'Smartcard Number', 'Smartcard Type', 'Transaction Type'],
      dtype='object')